# Autograd

Training a network means adjusting weights by the gradient of the loss with respect to each weight. `torch.autograd` computes those gradients for any graph of operations.

While the forward pass runs, PyTorch records every operation. The backward pass walks that record in reverse and applies the chain rule.

Everything here works the same on CPU, MPS and CUDA. This notebook stays on CPU so the numbers are reproducible.

In [ ]:
import torch

torch.manual_seed(0)

## A one-layer network

Input `x`, target `y`, weight `w` and bias `b`. Only `w` and `b` need gradients, because only they get updated. The input is data and is never optimized.

In [ ]:
x = torch.ones(5)
y = torch.zeros(3)

w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)

z = torch.matmul(x, w) + b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

print(x.shape, w.shape, b.shape, z.shape)
print(loss.shape, loss.item())

## The graph

`requires_grad` spreads forward: any result touching a tensor that needs gradients also needs them.

A **leaf** is a tensor created by the user rather than by an operation. Only leaves with `requires_grad=True` get a populated `.grad`.

`grad_fn` is the function that will be used to go backward through that step. Leaves have none.

In [ ]:
print(x.requires_grad, w.requires_grad, z.requires_grad)
print(x.is_leaf, w.is_leaf, z.is_leaf)
print(z.grad_fn)
print(loss.grad_fn)

## Computing gradients

`loss.backward()` fills `.grad` on every leaf that needs it. The gradient always has the same shape as the tensor it belongs to, which is a fast way to check that things line up.

In [ ]:
loss.backward()

print(w.grad.shape, w.shape)
print(b.grad.shape, b.shape)
print(w.grad)

The result is not magic. For this loss the gradient can be written by hand and compared.

The derivative of the loss with respect to `z` is `(sigmoid(z) - y) / n`, where `n` is the number of elements, because the default reduction is the mean. From there the chain rule gives `b` directly and `w` as the outer product with the input.

In [ ]:
grad_z = (torch.sigmoid(z) - y) / z.numel()
grad_w = torch.outer(x, grad_z)
grad_b = grad_z

print(torch.allclose(w.grad, grad_w))
print(torch.allclose(b.grad, grad_b))

## The graph is freed after backward

By default the record is thrown away once it has been used. A second `backward()` on the same graph raises, unless `retain_graph=True` was passed.

This is why the graph is rebuilt from scratch on every training iteration.

In [ ]:
try:
    loss.backward()
except RuntimeError as e:
    print(type(e).__name__)
    print(str(e)[:80])

## Gradients accumulate

`backward()` adds to `.grad` instead of replacing it. Three passes without clearing give three times the gradient.

In [ ]:
w2 = torch.ones(3, requires_grad=True)

for _ in range(3):
    out = (w2 * 2).sum()
    out.backward()
    print(w2.grad)

Clearing between steps gives the same gradient every time. In a real loop this is `optimizer.zero_grad()`.

Forgetting it is a classic bug: the model still trains, but each step uses the sum of all previous gradients, so the updates are wrong and grow over time.

In [ ]:
w3 = torch.ones(3, requires_grad=True)

for _ in range(3):
    out = (w3 * 2).sum()
    out.backward()
    print(w3.grad)
    w3.grad = None

## Turning tracking off

Two ways, used for different reasons:

- `torch.no_grad()` for a block, used during evaluation and inference. No graph is built, so it is faster and uses less memory.
- `.detach()` for a single tensor, used when a value has to leave the graph, for example to store or plot it.

In [ ]:
z = torch.matmul(x, w) + b
print(z.requires_grad)

with torch.no_grad():
    z_nograd = torch.matmul(x, w) + b
print(z_nograd.requires_grad)

z_det = z.detach()
print(z_det.requires_grad)

The other use is freezing. A parameter with `requires_grad=False` gets no gradient, so the optimizer cannot change it. That is how feature extraction works in transfer learning.

In [ ]:
frozen = torch.randn(5, 3, requires_grad=True)
frozen.requires_grad_(False)

out = (torch.matmul(x, frozen)).sum()

print(frozen.requires_grad)
print(out.requires_grad)
print(out.grad_fn)

## backward on a non-scalar

`backward()` with no argument only works when the output is a single number, which is why a loss is always reduced to a scalar.

For a non-scalar output PyTorch needs to know how to weight each element, so a tensor of the same shape must be passed.

In [ ]:
v = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
out = v * v

try:
    out.backward()
except RuntimeError as e:
    print(type(e).__name__)

out.backward(torch.ones_like(out))
print(v.grad)

The derivative of `v * v` is `2v`, so `[1, 2, 3]` gives `[2, 4, 6]`.

With loss, gradients and clearing in place, the only missing piece of a training loop is the optimizer that applies the update.